In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io.wavfile as wavfile
from scipy.signal import ShortTimeFFT, windows
from scipy.fft import fft
from sklearn.cluster import DBSCAN
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing

def create_spectrogram(audio, fs):
    w_s = fs * 10
    std = w_s // 6
    w = windows.gaussian(w_s, std=std)
    hop = fs // 2
    N = len(audio) // fs # audio length [s]

    SFT = ShortTimeFFT(win=w, hop=hop, fs=fs, scale_to='magnitude')
    Sx = SFT.spectrogram(audio)
    Sx_dB = 10 * np.log10(Sx)

    freqs = np.arange(Sx.shape[0]) * fs / w_s
    band_mask = (99.8 <= freqs) & (freqs <= 100.2)
    Sx_dB_filtered = Sx_dB[band_mask, :]

    width = 6.4 * N / 490
    fig = plt.figure(frameon=False, figsize=(width, 4.8))
    ax = fig.add_axes([0, 0, 1, 1])
    plt.imshow(
        Sx_dB_filtered,
        origin='lower',
        aspect='auto',
        cmap='magma',
        interpolation='bilinear'
    )
    ax.axis('off')
    fig.canvas.draw()
    img = np.asarray(fig.canvas.buffer_rgba())
    plt.close(fig)
    return img, int(width*100)

def color_convert(c):
    r, g, b, _ = c
    lumination = int(0.2126*r + 0.7152*g + 0.0722*b)
    return lumination

def img2array(img):
    height, width, _= img.shape
    xx, yy = [], []
    for y in range(height-1, -1, -1):
        for x in range(0, width):
            xx.append(x)
            yy.append(y)
    cords = np.column_stack((xx, yy))
    img_flat = img.reshape(-1, 4)
    colors = [color_convert(tuple(row)) for row in img_flat]
    X = np.column_stack((cords, colors))
    return X

def remove_outliers_iqr(data, axis, outlier_axis=2):
    min = np.min(data[:, axis])
    max = np.max(data[:, axis])
    sums = 0
    for v in range(min, max+1):
        line_mask = data[:, axis] == v
        if np.sum(line_mask) > 0:
            q1, q3 = np.percentile(data[line_mask, outlier_axis], [25, 75])
            iqr = q3 - q1
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr
            outliers_mask = line_mask & ((data[:, outlier_axis] < lower_bound) | (data[:, outlier_axis] > upper_bound))
            sums += np.sum(outliers_mask)
            data = data[outliers_mask == False]
    return data

def remove_outliers_DBSCAN(data, eps, min_samples):
    metric = 'euclidean'
    model = DBSCAN(eps = eps, min_samples = min_samples, metric = metric)
    model.fit(data[:, [0, 1]])
    label_mask = model.labels_ != -1
    data = data[label_mask]
    return data

def filter(X):
    mean_color = X[:, 2].mean()
    X = X[X[:, 2] > mean_color * 1.1, :]
    return X

def clean(X):
    for _ in range(5):
        X = remove_outliers_iqr(X, 0, 2)
        X = remove_outliers_iqr(X, 1, 2)
        X = remove_outliers_iqr(X, 1, 0)
    return X
    
def average_frequency(X):
    time, indices = np.unique(X[:, 0], return_inverse = True)
    means = np.bincount(indices, weights=X[:, 1]) / np.bincount(indices)
    return pd.Series(means, index=time.astype(int))

def smooth_chart(enf_ts):
    ses_model = SimpleExpSmoothing(enf_ts, initialization_method='heuristic').fit(smoothing_level=0.08, optimized=False)
    S = ses_model.fittedvalues
    return S

def fill_missing(n, X):
    missing_idx = list(set(np.arange(n)) - set(X.index))
    for idx in missing_idx:
        X.loc[idx] = float('NaN')
    X = X.sort_index()
    X = X.interpolate(method="nearest", limit_direction="both")
    X = X.ffill().bfill()

    return X


In [ ]:
# takes 1.5h
def large_precompute(import_path, export_path, export_chunk_path="./precomp-data/h1-ref-day/chunks/"):
    fs, audio = wavfile.read(import_path)        
    N = len(audio)
    C = int(2e5) #chunk size
    enf = pd.Series()

    for i in range((N+C-1) // C):
        print(f'Precomputing chunk {i}')
        audio_part = audio[i*C:(i+1)*C]
        img, width = create_spectrogram(audio_part, fs)
        X = img2array(img)
        print(f'Created spectrogram {i}')

        X_filtered = filter(X.copy())
        X_cleaned = X_filtered.copy()
        X_grouped = X_cleaned.copy()
        X_cleaned = clean(X_filtered.copy())
        X_grouped = remove_outliers_DBSCAN(X_cleaned.copy(), 11, 100)
        X_averaged = average_frequency(X_grouped)
        X_filled = fill_missing(width, X_averaged)
        print(f'Cleaned {i}')

        enf_smoothed = smooth_chart(X_filled)
        enf_smoothed.to_csv(f'{export_chunk_path}/chunk_{i}.csv') # backup for precomputed chunk

        enf = pd.concat([enf, enf_smoothed], ignore_index=True)
    
    enf.to_csv(export_path)


In [ ]:
def file_name(i, tail):
    name = ""
    if i < 10:
        name = '00' + f'{i}'
    elif i < 100:
        name = '0' + f'{i}'
    else:
        name = f'{i}'
    return name + tail

In [ ]:
import_path = f"./data/h1-ref-day/day_001_ref.wav"
export_path = f"./precomp-data/h1-ref-day/day_001_ref_precomp.csv"
large_precompute(import_path, export_path)

Precomputing chunk 0
Created spectrogram 0
Cleaned 0
Precomputing chunk 1


/tmp/ipykernel_10233/1675985146.py:28: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  pd.concat([enf, enf_smoothed], ignore_index=True)


Created spectrogram 1
Cleaned 1
Precomputing chunk 2


/tmp/ipykernel_10233/1675985146.py:28: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  pd.concat([enf, enf_smoothed], ignore_index=True)


Created spectrogram 2
Cleaned 2
Precomputing chunk 3


...

Created spectrogram 171
Cleaned 171
Precomputing chunk 172


/tmp/ipykernel_10233/1675985146.py:28: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  pd.concat([enf, enf_smoothed], ignore_index=True)


Created spectrogram 172
Cleaned 172


/tmp/ipykernel_10233/1675985146.py:28: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  pd.concat([enf, enf_smoothed], ignore_index=True)


IsADirectoryError: [Errno 21] Is a directory: './precomp-data/h1-ref-day/chunks/'

In [ ]:
enf = pd.Series()
for i in range(1, 173):
    chunk = pd.read_csv(f"./precomp-data/h1-ref-day/chunks/chunk_{i}", index_col=0).squeeze()
    enf = pd.concat([enf, chunk], ignore_index=True)
enf
export_path = f"./precomp-data/h1-ref-day/day_001_ref_precomp.csv"
enf.to_csv(export_path)


/tmp/ipykernel_10233/567735122.py:4: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  enf = pd.concat([enf, chunk], ignore_index=True)
